# detfuse — Detector 評估

比較 **KeywordDetector**（L1 軟分數）、**Qwen2.5-0.5B**（L2 信心分數）與**並行融合**（α × L1 + (1-α) × L2）在偵測「免費食物」貼文的準確率。

執行環境：Colab（T4 GPU）

## 1. 安裝依賴

In [1]:
!pip install -q transformers accelerate peft

## 2. KeywordDetector（直接複製 detector.py 的 L1 邏輯）

In [2]:
import re

# 與 detector.py 保持一致
_FREE_WORDS = r'免費|free|請拿|拿走|多餘|多的|送人|不要了|剩食|剩菜|拿去|有需要|帶走|送出|分享'
_FOOD_WORDS = r'食物|食品|飯|麵|便當|零食|餅乾|水果|蔬菜|菜|湯|肉|蛋|麵包|吐司|料理|點心|糕|餅|粽|飲料|奶茶|咖啡|茶|寶特瓶|三明治|沙拉|漢堡|披薩|壽司|飯糰|泡麵|湯圓'
_PATTERN_FREE_FOOD    = re.compile(rf'(?=.*({_FREE_WORDS}))(?=.*({_FOOD_WORDS}))', re.IGNORECASE)
_PATTERN_NOT_FOOD     = re.compile(r'免費.*?(?:課程|諮詢|活動|講座|workshop|票|名額|參加|索取)', re.IGNORECASE)
_PATTERN_EVENT_FOOD   = re.compile(
    r'(?:研討會|活動|工作坊|演講|說明會|工作人員).{0,10}(?:便當|餐盒|餐點|飲料|食物|點心)',
    re.IGNORECASE,
)
_PATTERN_HAS_CATERING = re.compile(r'(?:有|免費|提供)供餐', re.IGNORECASE)


def keyword_detect(text: str) -> bool:
    if not text:
        return False
    if _PATTERN_NOT_FOOD.search(text):
        return False
    return bool(_PATTERN_FREE_FOOD.search(text))


def l1_score(text: str) -> float:
    """L1 軟分數，[0, 1]。"""
    score = 0.0
    if _PATTERN_NOT_FOOD.search(text):
        return 0.0
    if _PATTERN_FREE_FOOD.search(text):     score += 0.80
    if _PATTERN_EVENT_FOOD.search(text):    score += 0.60
    if _PATTERN_HAS_CATERING.search(text):  score += 0.50
    has_free = bool(re.search(_FREE_WORDS, text, re.IGNORECASE))
    has_food = bool(re.search(_FOOD_WORDS, text, re.IGNORECASE))
    if has_free and not _PATTERN_FREE_FOOD.search(text): score += 0.20
    if has_food and not _PATTERN_FREE_FOOD.search(text): score += 0.10
    return min(score, 1.0)


print('L1 patterns + l1_score() loaded')

L1 patterns + l1_score() loaded


## 3. 測試資料集

由 Claude Code 標記，來源：`data/categories/free_food/`

- **training_data.json**：894 筆（正例 212 / 負例 682）
- **test_data.json**：234 筆（正例 58 / 負例 176）

評估使用 `test_data.json`，欄位：`text`、`label`（1 = 免費食物，0 = 非）

In [3]:
import json, urllib.request

BRANCH = 'feature/parallel-fusion'
BASE_URL = f'https://raw.githubusercontent.com/syoslyot/detfuse/{BRANCH}/data/categories/free_food'

def load_samples(filename):
    url = f'{BASE_URL}/{filename}'
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read())
    return [(d['label'], d['text']) for d in data]

TRAIN_SAMPLES = load_samples('training_data.json')
TEST_SAMPLES  = load_samples('test_data.json')
SAMPLES = TEST_SAMPLES  # 評估用 test set

print(f'training: {sum(l==1 for l,_ in TRAIN_SAMPLES)} 正例 / {sum(l==0 for l,_ in TRAIN_SAMPLES)} 負例')
print(f'test:     {sum(l==1 for l,_ in TEST_SAMPLES)} 正例 / {sum(l==0 for l,_ in TEST_SAMPLES)} 負例')

training: 212 正例 / 682 負例
test:     58 正例 / 176 負例


In [4]:
import datetime

# 計算 report/experiment/ 現有報告數，自動產生本次實驗 ID
_api = "https://api.github.com/repos/syoslyot/detfuse/contents/report/experiment"
try:
    with urllib.request.urlopen(_api) as _r:
        _files = json.loads(_r.read())
    _n = len([f for f in _files if f["name"].startswith("experiment_") and f["name"].endswith(".md")])
except:
    _n = 0

EXPERIMENT_ID = f"experiment_{_n + 1:02d}"
RUN_DATE = datetime.datetime.now().strftime("%Y-%m-%d")
print(f"實驗 ID：{EXPERIMENT_ID}  日期：{RUN_DATE}")

實驗 ID：experiment_03  日期：2026-05-30


## 4. 評估 KeywordDetector

In [5]:
def evaluate(name, predict_fn, samples):
    tp = fp = tn = fn = 0
    errors = []
    for label, text in samples:
        pred = predict_fn(text)
        if label == 1 and pred:     tp += 1
        elif label == 0 and not pred: tn += 1
        elif label == 0 and pred:
            fp += 1
            errors.append(('FP', text[:60]))
        else:
            fn += 1
            errors.append(('FN', text[:60]))

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f'\n── {name} ──')
    print(f'  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}')
    print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    if errors:
        print('  錯誤案例:')
        for tag, t in errors:
            print(f'    [{tag}] {t}')
    return dict(name=name, precision=precision, recall=recall, f1=f1)

kw_result = evaluate('KeywordDetector (L1)', keyword_detect, SAMPLES)


── KeywordDetector (L1) ──
  Precision: 78.05%  Recall: 55.17%  F1: 64.65%
  TP=32 FP=9 TN=167 FN=26
  錯誤案例:
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FN] 研討會哈蜜瓜便當
座標成大光復校區中文系館演講廳（大門口）
研討會哈蜜瓜便當
座標成大光復校區中文系館演講廳（大門口）

    [FN] (發完囉!)
研討會便當-社科院南棟一樓
有需要的人歡迎來領取（食安自負）
    [FN] 營隊剩下的熏雞吐司
在社科院80103 食安自負
麻煩最後一個拿完的人留言

陳證仰
營隊剩下的熏雞吐司
在社科院801
    [FP] 成大中文系館（成功校區）
研討會便當免費領
5個葷食 3個素食
成大中文系館（成功校區）
研討會便當免費領
5個葷食 3
    [FN] #更 已發完謝謝大家
外文研討會多出來的餐盒
讓你早餐吃飽飽～
放在成功湖邊（修齊大樓門口）的石桌上
歡迎自取
    [FN] 研討會七飯亭便當34個
座標成大光復校區中文系演講廳（大門口）
食安自負
研討會七飯亭便當34個
座標成大光復校區中文系
    [FN] 系統系研討會免費餐盒（已發完）
拿幾個都可以，先拿先贏！
系統系研討會免費餐盒（已發完）
拿幾個都可以，先拿先贏！
免費
    [FN] 資源系研討會好吃的午餐，有素食便當。
要自備餐具
在舊系館進來後右轉，到2:20要來要快。
資源系研討會好吃的午餐，有素
    [FN] 中午活動剩下的便當跟飲料（需自備環保杯），在光復校區國際會議廳請自取，到16:45，食安自負（都在冷氣房裡）。
免費
 
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FN] 超級好吃便當還有三分春色的飲料、啊品項很多啦自己挑，放在藝研所東側門，食安自負！
$1
 

（已售出） 來吃飯

（已
    [FN] 活動剩下很多
 熱咖啡 
 熱奶茶 
 梅子綠 
熱麥茶
快來哦！歡迎裝爆
在國際會議廳 一活多功能廳這裡！
我們也會去
    [FN]

## 5. Qwen2.5-0.5B-Instruct（L2 模型，需 GPU）

模擬 `OllamaDetector` 的邏輯，但改用 HuggingFace Transformers 直接跑。
這樣不需要 Ollama server，在 Colab 上也能驗證模型品質。

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
print(f'模型載入完成，裝置：{next(model.parameters()).device}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型載入完成，裝置：cuda:0


In [7]:
PROMPT_TMPL = (
    '判斷以下貼文是否在提供免費食物或飲料（可以現在就去拿）。\n'
    '請只輸出 0 到 9 的整數，代表信心程度（0 = 完全不是，9 = 完全確定是）。不要輸出其他任何文字。\n\n'
    '貼文：{text}'
)


def qwen_prob(text: str) -> float:
    """回傳 [0, 1] 信心分數（0-9 digit / 9.0）。"""
    messages = [
        {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    ids = ids.to(model.device)
    input_len = ids.shape[1]
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    for ch in answer:
        if ch.isdigit():
            return int(ch) / 9.0
    return 0.5  # 無法解析 → 中立


def qwen_detect(text: str) -> bool:
    return qwen_prob(text) > 0.5


# Smoke test
print(qwen_prob('有多的便當，免費拿走，在工程館'))   # expect ≥ 0.5
print(qwen_prob('出售二手書'))                        # expect < 0.5

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


0.7777777777777778
0.1111111111111111


In [8]:
qwen_result = evaluate('Qwen2.5-0.5B (L2)', qwen_detect, SAMPLES)


── Qwen2.5-0.5B (L2) ──
  Precision: 14.29%  Recall: 22.41%  F1: 17.45%
  TP=13 FP=78 TN=98 FN=45
  錯誤案例:
    [FN] 《免費便當、沙拉、湯》（沒了）
系展剩下的，歡迎來成功校區地科系館1F自取
可以順便來看地科系展
《免費便當、沙拉、湯》
    [FP] 寒假的營隊在光餐的剩食
讓阿姨見識到二手版的威力
所以認識的阿姨請我幫忙宣導
光復自助餐的半價時段
星期一~四 18：3
    [FP] 更/拿完了謝謝各位
午安您好，又到了午飯時間。
/一樣建築系館旁的遮陽地方/
今日菜單
刈包+碗粿*4
刈包+米糕*5

    [FP] 【出售】大同 TATUNG 51L 經典小冰箱
• 商品名稱： 大同冷藏電冰箱 (TR-50HC)
• 商品尺寸： 寬 
    [FP] 是誰把我的帥氣老婆弄倒了⋯
下午騎車發現左邊煞車凹損
車體左側有多處擦傷
5/16（六）
8:40-16:10 停放於雲
    [FN] 光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
免費
    [FP] 出售未使用到的商品～
1. One meter輕巧負離子高速吹風機 $1500（全新）- 原價2980
2. KD203
    [FP] 徵好心學弟妹幫我買香格里拉杜拜巧克力Q餅到Costco附近
跑腿費250
+一杯飲料實報實銷
時間是這週五下午
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FP] ［預定中］
大四畢業出清！
【GIANT momentum 腳踏車 售$4000】
附贈：車燈+鎖
大二時於捷安特門市購
    [FP] 更：找到了謝謝好心人qqqqqqqqqq
抱歉打擾
11/26 12:00左右 （中午時段）
走去醫學院的路上錢包不見了
    [FN] 營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
免費
 

登大人多
    [FN] 研討會哈蜜瓜便當
座標成大光復校區中文系館演講廳（大門口）
研討會哈蜜瓜便當
座標成大光復校區中文

## 6. 並行融合（α × L1 + (1-α) × L2）

L1 和 L2 各自計算分數，再加權融合——兩層都有發言權，都能糾正對方的誤判。

In [9]:
def fuse(text: str, alpha: float = 0.35, threshold: float = 0.50) -> bool:
    s1 = l1_score(text)
    s2 = qwen_prob(text)
    return (alpha * s1 + (1 - alpha) * s2) > threshold


fusion_result = evaluate('Fusion α=0.35 τ=0.50', fuse, SAMPLES)


── Fusion α=0.35 τ=0.50 ──
  Precision: 19.79%  Recall: 32.76%  F1: 24.68%
  TP=19 FP=77 TN=99 FN=39
  錯誤案例:
    [FN] 《免費便當、沙拉、湯》（沒了）
系展剩下的，歡迎來成功校區地科系館1F自取
可以順便來看地科系展
《免費便當、沙拉、湯》
    [FP] 寒假的營隊在光餐的剩食
讓阿姨見識到二手版的威力
所以認識的阿姨請我幫忙宣導
光復自助餐的半價時段
星期一~四 18：3
    [FP] 更/拿完了謝謝各位
午安您好，又到了午飯時間。
/一樣建築系館旁的遮陽地方/
今日菜單
刈包+碗粿*4
刈包+米糕*5

    [FP] 【出售】大同 TATUNG 51L 經典小冰箱
• 商品名稱： 大同冷藏電冰箱 (TR-50HC)
• 商品尺寸： 寬 
    [FP] 是誰把我的帥氣老婆弄倒了⋯
下午騎車發現左邊煞車凹損
車體左側有多處擦傷
5/16（六）
8:40-16:10 停放於雲
    [FN] 光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
免費
    [FP] 出售未使用到的商品～
1. One meter輕巧負離子高速吹風機 $1500（全新）- 原價2980
2. KD203
    [FP] 徵好心學弟妹幫我買香格里拉杜拜巧克力Q餅到Costco附近
跑腿費250
+一杯飲料實報實銷
時間是這週五下午
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FP] ［預定中］
大四畢業出清！
【GIANT momentum 腳踏車 售$4000】
附贈：車燈+鎖
大二時於捷安特門市購
    [FP] 更：找到了謝謝好心人qqqqqqqqqq
抱歉打擾
11/26 12:00左右 （中午時段）
走去醫學院的路上錢包不見了
    [FN] 營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
免費
 

登大人多
    [FP] **協助轉發**
填問卷拿100元全家禮券！最高400元（不用抽獎）
大家好！這是南應大商品

## 7. 比較結果

In [10]:
print(f'\n{"模型":<28} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 62)
for r in [kw_result, qwen_result, fusion_result]:
    print(f"{r['name']:<28} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")


模型                            Precision     Recall         F1
--------------------------------------------------------------
KeywordDetector (L1)             78.05%     55.17%     64.65%
Qwen2.5-0.5B (L2)                14.29%     22.41%     17.45%
Fusion α=0.35 τ=0.50             19.79%     32.76%     24.68%


## 8. Fine-tune Qwen2.5-0.5B（LoRA）

用 `TRAIN_SAMPLES`（894 筆）對模型做 LoRA fine-tune，格式與 inference prompt 一致。
訓練完後直接在同一 session 重新評估，比較 fine-tune 前後差異。

In [11]:
from torch.utils.data import Dataset as TorchDataset
from peft import LoraConfig, get_peft_model

class SFTDataset(TorchDataset):
    def __init__(self, samples, tokenizer, max_length=512):
        self.examples = []
        for label, text in samples:
            digit = '9' if label == 1 else '0'
            prompt_messages = [
                {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
                {'role': 'user',   'content': PROMPT_TMPL.format(text=text[:400])},
            ]
            full_messages = prompt_messages + [{'role': 'assistant', 'content': digit}]

            # Tokenize prompt only to find where assistant reply starts
            prompt_text = tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
            )
            prompt_len = len(tokenizer(prompt_text)['input_ids'])

            # Tokenize full conversation
            full_text = tokenizer.apply_chat_template(full_messages, tokenize=False)
            enc = tokenizer(full_text, truncation=True, max_length=max_length)
            input_ids = enc['input_ids']

            # Only compute loss on assistant reply tokens
            labels_ids = [-100] * prompt_len + input_ids[prompt_len:]

            self.examples.append({
                'input_ids':      torch.tensor(input_ids),
                'attention_mask': torch.tensor(enc['attention_mask']),
                'labels':         torch.tensor(labels_ids),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

train_dataset = SFTDataset(TRAIN_SAMPLES, tokenizer)
print(f'訓練樣本數：{len(train_dataset)}')

# Verify loss masking
sample = train_dataset[0]
n_label_tokens = (sample['labels'] != -100).sum().item()
print(f'Total tokens: {len(sample["input_ids"])}, Label tokens (assistant only): {n_label_tokens}')
# 預期：label tokens 應只有 1–3 個，遠少於 total tokens

訓練樣本數：894
Total tokens: 254, Label tokens (assistant only): 3


In [12]:
import subprocess, importlib, sys
subprocess.run(['sed', '-i',
    's/if torchao_version < TORCHAO_MINIMUM_VERSION:/if False:  # patched/',
    '/usr/local/lib/python3.12/dist-packages/peft/import_utils.py'])
for m in list(sys.modules.keys()):
    if 'peft' in m: del sys.modules[m]

from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

EPOCHS, BATCH_SIZE, LR = 3, 4, 2e-4

def collate_fn(batch):
    max_len = max(x['input_ids'].shape[0] for x in batch)
    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    attn_mask = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        l = x['input_ids'].shape[0]
        input_ids[i, :l] = x['input_ids']
        attn_mask[i, :l] = x['attention_mask']
        labels[i, :l]    = x['labels']   # use pre-computed masked labels
    return {'input_ids': input_ids, 'attention_mask': attn_mask, 'labels': labels}

loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
total_steps  = len(loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
optimizer  = AdamW(model.parameters(), lr=LR)
scheduler  = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler     = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if (step + 1) % 20 == 0:
            print(f'Epoch {epoch+1} step {step+1}/{len(loader)} loss={total_loss/(step+1):.4f}')
    print(f'Epoch {epoch+1} avg loss: {total_loss/len(loader):.4f}')

print('Fine-tune 完成')

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


/tmp/ipykernel_1720/2553299591.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler     = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_1720/2553299591.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_1720/2553299591.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1 step 20/224 loss=0.4310
Epoch 1 step 40/224 loss=0.2863
Epoch 1 step 60/224 loss=0.2414
Epoch 1 step 80/224 loss=0.2205
Epoch 1 step 100/224 loss=0.2003
Epoch 1 step 120/224 loss=0.1891
Epoch 1 step 140/224 loss=0.1738
Epoch 1 step 160/224 loss=0.1659
Epoch 1 step 180/224 loss=0.1615
Epoch 1 step 200/224 loss=0.1535
Epoch 1 step 220/224 loss=0.1451
Epoch 1 avg loss: 0.1433
Epoch 2 step 20/224 loss=0.0733
Epoch 2 step 40/224 loss=0.0873
Epoch 2 step 60/224 loss=0.0750
Epoch 2 step 80/224 loss=0.0611
Epoch 2 step 100/224 loss=0.0640
Epoch 2 step 120/224 loss=0.0643
Epoch 2 step 140/224 loss=0.0591
Epoch 2 step 160/224 loss=0.0550
Epoch 2 step 180/224 loss=0.0537
Epoch 2 step 200/224 loss=0.0553
Epoch 2 step 220/224 loss=0.0561
Epoch 2 avg loss: 0.0561
Epoch 3 step 20/224 loss=0.0347
Epoch 3 step 40/224 loss=0.0343
Epoch 3 step 60/224 loss=0.0302
Epoch 3 step 80/224 loss=0.0306
Epoch 3 step 100/224 loss=0.0277
Epoch 3 step 120/224 loss=0.0307
Epoch 3 step 140/224 loss=0.0290
Epoch

## 8.5 上傳到 HuggingFace Hub

登入後將 fine-tuned LoRA adapter 推上去，之後任何地方都能用名稱載入。

In [18]:
from huggingface_hub import notebook_login
notebook_login()  # 貼上 HuggingFace Write token


In [19]:
REPO_NAME = 'syoslyot/qwen-detfuse-finetuned'
_msg = f"{EXPERIMENT_ID} ({RUN_DATE})"

model.push_to_hub(REPO_NAME, commit_message=_msg)
tokenizer.push_to_hub(REPO_NAME, commit_message=_msg)
print(f"上傳完成：{_msg}")
print(f"https://huggingface.co/{REPO_NAME}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  13%|#2        |  543kB / 4.34MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mps56mb0c8/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


上傳完成：experiment_03 (2026-05-30)
https://huggingface.co/syoslyot/qwen-detfuse-finetuned


## 9. 重新評估（fine-tune 後）

In [20]:
model.eval()

qwen_ft_result  = evaluate('Qwen2.5-0.5B fine-tuned (L2)', qwen_detect, SAMPLES)
fusion_ft_result = evaluate('Fusion ft α=0.35 τ=0.50', fuse, SAMPLES)

print(f'\n{"模型":<32} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 66)
for r in [kw_result, qwen_result, fusion_result, qwen_ft_result, fusion_ft_result]:
    print(f"{r['name']:<32} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")


── Qwen2.5-0.5B fine-tuned (L2) ──
  Precision: 85.29%  Recall: 100.00%  F1: 92.06%
  TP=58 FP=10 TN=166 FN=0
  錯誤案例:
    [FP] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FP] 成大中文系館（成功校區）
研討會便當免費領
5個葷食 3個素食
成大中文系館（成功校區）
研討會便當免費領
5個葷食 3
    [FP] 新生營多的便當 在電機系館一樓
最後拿的講一下感謝
新生營多的便當 在電機系館一樓
最後拿的講一下感謝
免費
 

便當
    [FP] 更 沒了 抱歉撲空的人嗚嗚
雨停了該吃午餐囉
圓形是燒肉 方形一個素食其他排骨便當
社科北棟二樓樓梯口 這次還有餐具可以
    [FP] 講座結束剩餘～在國際會議廳前面廣場！！
免費
 

（已售出） 免費高級好吃餐盒

（已售出）  免費高級好吃餐盒
免費
    [FP] 講座結束剩餘～在國際會議廳前面廣場！！
免費
 

（已售出） 免費高級好吃餐盒

（已售出）  免費高級好吃餐盒
免費
    [FP] 免費
 

（已售出） 社團博覽會｜成大社聯會ig追蹤換免費飲料

（已售出）  社團博覽會｜成大社聯會ig追蹤換免費飲
    [FP] 講座結束剩餘～在國際會議廳前面廣場！！
免費
 

（已售出） 免費高級好吃餐盒

（已售出）  免費高級好吃餐盒
免費
    [FP] 電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿的留言，感謝！
電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿
    [FP] 活動剩下的！
超級好喝
大家趕快帶水瓶來裝！
［小米奶茶、冰橙紅玉、多多綠］
在格致廳大講堂

── Fusion ft α=0.35 τ=0.50 ──
  Precision: 85.29%  Recall: 100.00%  F1: 92.06%
  TP=58 FP=10 TN=166 FN=0
  錯誤案例:
    [FP] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FP] 成大